# ISLES 2015 SISS reproduction entry point

This notebook runs entirely in Google Colab. Nothing needs to be installed on your own computer.

**Current milestone:** download and validate ISLES 2015, discover the 28 labeled SISS cases, load one FLAIR volume, and verify the reconstructed 9-feature extractor. Model training is added in the next milestone.


In [ ]:
!rm -rf HybridStrokeSeg-Reproduction
!git clone -q https://github.com/NTinkicht/HybridStrokeSeg-Reproduction.git
%cd HybridStrokeSeg-Reproduction
!python -m pip install -q -e '.[notebook]'


## 1. Download the organizer re-archive
The downloader verifies the published MD5 checksum before extraction.


In [ ]:
!python scripts/download_isles2015.py --extract


## 2. Discover and validate labeled SISS cases


In [ ]:
from pathlib import Path
from hybridstrokeseg.data import discover_siss_cases

root = Path('data/raw/isles2015')
cases = discover_siss_cases(root)
print('Complete labeled cases:', len(cases))
for case in cases[:5]:
    print(case.case_id)

assert len(cases) == 28, f'Expected 28 SISS training cases, found {len(cases)}'


## 3. Load a FLAIR volume and lesion mask


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hybridstrokeseg.data import load_volume

case = cases[0]
flair, flair_meta = load_volume(case.flair)
mask, mask_meta = load_volume(case.ot)
print('FLAIR shape (z,y,x):', flair.shape)
print('Spacing xyz:', flair_meta['spacing_xyz'])

# For visualization only, choose the slice with the largest labeled lesion area.
# This is NOT yet adopted as the paper-reproduction slice-selection rule.
slice_index = int(np.argmax((mask > 0).sum(axis=(1, 2))))
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(flair[slice_index], cmap='gray')
ax.contour(mask[slice_index] > 0, levels=[0.5], linewidths=1)
ax.set_title(f'{case.case_id} - slice {slice_index} (visualization only)')
ax.axis('off');


## 4. Reconstruct the paper's nine handcrafted features


In [ ]:
from hybridstrokeseg import FeatureConfig, extract_nine_features, feature_names

cfg = FeatureConfig(threshold=20.0, run_window=25, coordinate_mode='raw')
X9 = extract_nine_features(flair[slice_index], cfg)
print('Feature tensor:', X9.shape)
print('Feature order:', feature_names())
assert X9.shape[-1] == 9


## Reproducibility warning
The paper does not specify enough detail to recover the exact weighted-mean kernel or exact directional 'run-length' operation. The choices executed above are explicit clean-room assumptions documented in `docs/reconstruction_assumptions.md`; they must not be described as original author code.
